In [1]:
import os
import re
import pandas as pd

In [2]:
RESULTS_DIR = "./results"

In [3]:
def extract_accuracy(file_path):
    """
    Extrae Acc=XX.XX del archivo HResults.
    """
    with open(file_path, "r") as f:
        content = f.read()

    match = re.search(r'Acc=([0-9]+\.[0-9]+)', content)

    if match:
        return float(match.group(1))
    else:
        raise ValueError(f"Accuracy not found in {file_path}")

In [4]:
state_dirs = sorted([
    d for d in os.listdir(RESULTS_DIR)
    if re.match(r"results_\d+_states", d)
])

print("Detected state models:")
for s in state_dirs:
    print(f"  {s}")

Detected state models:
  results_3_states
  results_4_states
  results_5_states
  results_6_states
  results_7_states


In [5]:
for state_dir in state_dirs:

    state_match = re.search(r"results_(\d+)_states", state_dir)
    STATES = state_match.group(1)

    RESULTS_ROOT = os.path.join(RESULTS_DIR, state_dir)

    OUTPUT_EXCEL = os.path.join(
        RESULTS_ROOT,
        f"HMM_Results_{STATES}_states.xlsx"
    )

    print(f"\nGenerating {OUTPUT_EXCEL}")

    with pd.ExcelWriter(OUTPUT_EXCEL, engine="openpyxl") as writer:

        # =====================================================
        # DETECTAR GROUP_XX EXISTENTES
        # =====================================================

        outer_groups = sorted([
            g for g in os.listdir(RESULTS_ROOT)
            if re.match(r"Group_\d+", g)
        ])

        for outer_group in outer_groups:

            outer_num = re.search(r"Group_(\d+)", outer_group).group(1)
            outer_path = os.path.join(
                RESULTS_ROOT,
                outer_group
            )

            print(f"  Processing {outer_group}")

            # =================================================
            # DETECTAR SUBGRUPOS GroupXX_YY
            # =================================================

            inner_groups = sorted([
                g for g in os.listdir(outer_path)
                if re.match(
                    rf"Group{outer_num}_\d+",
                    g
                )
            ])

            if not inner_groups:
                print(f"    No inner groups in {outer_group}")
                continue

            # =================================================
            # DETECTAR TODAS LAS GAUSSIANAS EXISTENTES
            # =================================================

            gaussian_nums = set()

            for inner_group in inner_groups:

                inner_path = os.path.join(
                    outer_path,
                    inner_group
                )

                gaussian_dirs = [
                    g for g in os.listdir(inner_path)
                    if re.match(r"\d+_gaussians", g)
                ]

                for g in gaussian_dirs:
                    n = int(
                        re.search(r"(\d+)_gaussians", g).group(1)
                    )
                    gaussian_nums.add(n)

            gaussian_nums = sorted(gaussian_nums)

            print(
                f"    Found gaussian numbers: {gaussian_nums}"
            )

            # =================================================
            # CREAR DATAFRAME DINÁMICO
            # =================================================

            df = pd.DataFrame(
                index=inner_groups,
                columns=[
                    f"{g}_gaussians"
                    for g in gaussian_nums
                ],
                dtype=float
            )

            # =================================================
            # LEER RESULTADOS
            # =================================================

            for inner_group in inner_groups:

                inner_num = re.search(
                    rf"Group{outer_num}_(\d+)",
                    inner_group
                ).group(1)

                inner_path = os.path.join(
                    outer_path,
                    inner_group
                )

                for g in gaussian_nums:

                    results_file = os.path.join(
                        inner_path,
                        f"{g}_gaussians",
                        "HResults",
                        f"results{outer_num}_state{inner_num}.txt"
                    )

                    if os.path.exists(results_file):

                        acc = extract_accuracy(
                            results_file
                        )

                        df.loc[
                            inner_group,
                            f"{g}_gaussians"
                        ] = acc

                    else:
                        print(
                            f"      Missing: {results_file}"
                        )

            # =================================================
            # MEDIAS
            # =================================================

            df["mean"] = df.mean(axis=1)

            mean_row = df.mean(axis=0)

            df.loc["mean"] = mean_row

            df = df.round(2)

            # =================================================
            # GUARDAR HOJA
            # =================================================

            sheet_name = outer_group[:31]

            df.to_excel(
                writer,
                sheet_name=sheet_name
            )

    print(f"Excel generated: {OUTPUT_EXCEL}")

print("All Excel files were generated correctly.")


Generating ./results\results_3_states\HMM_Results_3_states.xlsx
  Processing Group_01
    Found gaussian numbers: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
  Processing Group_02
    Found gaussian numbers: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
  Processing Group_03
    Found gaussian numbers: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
  Processing Group_04
    Found gaussian numbers: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]
  Processing Group_05
    Found gaussian numbers: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
  Processing Group_06
    Found gaussian numbers: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
  Processing Group_07
    Found gaussian numbers: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
  Processing Group_08
    Found gaussian numbers: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
  Processing Group_09
    Found gaussian numbers: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
  Processing Group_10
    Found gaussian numbers: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
Excel generated: ./results\results_3_states\HMM_Results_3_states.xlsx

Generating ./results\res